# Indian Air Traffic Network Analysis

This notebook contains the main workflow for preprocessing the air-traffic data, building yearly networks, and analyzing their structure, robustness, and communities.


## Imports


In [ ]:
from copy import deepcopy
from pathlib import Path

import collections
import community as community_louvain
import geopandas as gpd
import matplotlib as mpl
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import powerlaw
from shapely.geometry import Point
from tqdm import tqdm

# plt.style.use('seaborn-v0_8-deep')
plt.style.use('fivethirtyeight')

pd.set_option('future.no_silent_downcasting', True)


## Data Preprocessing


### Required Functions


In [ ]:
def preprocess(df):
    df['CITY 1'] = df['CITY 1'].str.upper();
    df['CITY 2'] = df['CITY 2'].str.upper();
    df['CITY 1'] = df['CITY 1'].str.strip();
    df['CITY 1'] = df['CITY 1'].str.rstrip();
    df['CITY 2'] = df['CITY 2'].str.strip();
    df['CITY 2'] = df['CITY 2'].str.rstrip();
    df.replace('-', 0, inplace=True);
    df.replace('DEOGHAR AIRPORT', 'DEOGHAR', inplace=True);
    df.replace('HOLLONGI AIRPORT, ITANAGAR', 'ITANAGAR', inplace=True);
    df.replace('KUSHINAGAR', 'KUSHINAGAR INTERNATIONAL AIRPORT', inplace=True);
    df.replace('BIDAR AIRPORT, KARN', 'BIDAR AIRPORT, KARNATAKA', inplace=True);
    df.replace('KALABURAGI, KARNAT', 'KALABURAGI, KARNATAKA', inplace=True);
    df.replace('Itanagar', 'ITANAGAR', inplace=True);
    df.replace('PUDUCHERRY', 'PONDICHERRY', inplace=True);
    df['PASSENGERS FROM CITY 2'] = df['PASSENGERS FROM CITY 2'].astype(int);
    df['PASSENGERS TO CITY 2'] = df['PASSENGERS TO CITY 2'].astype(int);
    
    return None

In [ ]:
def graphi_format(df) -> pd.DataFrame:
    df1 = df.copy()
    df2 = df1.copy()
    
    df1.drop(columns=['PASSENGERS FROM CITY 2'], inplace=True)
    df2['CITY 1'], df2['CITY 2'] = df2['CITY 2'], df2['CITY 1']
    
    df2.drop(columns=['PASSENGERS TO CITY 2'], inplace=True)
    df2.rename(columns={'PASSENGERS FROM CITY 2': 'PASSENGERS TO CITY 2'}, inplace=True)
    
    return  pd.concat([df1, df2], ignore_index=True)

### Loading yearly data


In [ ]:
RAW_DATA_DIR = Path('../data/raw')

MONTH_CODES = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
MONTH_LABELS = {
    'JAN': 'January',
    'FEB': 'February',
    'MAR': 'March',
    'APR': 'April',
    'MAY': 'May',
    'JUN': 'June',
    'JUL': 'July',
    'AUG': 'August',
    'SEP': 'September',
    'OCT': 'October',
    'NOV': 'November',
    'DEC': 'December',
}
YEAR_MONTHS = {
    2015: ['APR', 'MAY', 'JUN', 'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC'],
    2016: MONTH_CODES,
    2017: MONTH_CODES,
    2018: MONTH_CODES,
    2019: MONTH_CODES,
    2020: MONTH_CODES,
    2021: MONTH_CODES,
    2022: MONTH_CODES,
    2023: MONTH_CODES,
}


def load_year_data(year):
    monthly_data = []
    for month in YEAR_MONTHS[year]:
        file_path = RAW_DATA_DIR / f'IndianAirways{year}' / f'{month}{year}.xlsx'
        month_data = pd.read_excel(file_path)
        preprocess(month_data)
        globals()[f'{month.lower()}{year}_data'] = month_data
        monthly_data.append(month_data)

    combined_data = pd.concat(monthly_data, ignore_index=True)
    combined_data = combined_data.groupby(['CITY 1', 'CITY 2'], as_index=False).sum()

    year_data = monthly_data + [combined_data]
    year_data_graphi_format = [graphi_format(data) for data in year_data]

    globals()[f'combined{year}_data'] = combined_data
    globals()[f'year{year}_data'] = year_data
    globals()[f'year{year}_data_graphi_format'] = year_data_graphi_format

    return year_data, year_data_graphi_format


yearly_data = {}
yearly_data_graphi_format = {}
for year in YEAR_MONTHS:
    yearly_data[year], yearly_data_graphi_format[year] = load_year_data(year)


### Assigning IDs, latitude, longitude, and GeoPandas geometry


In [ ]:
LOOKUP_PATH = Path('../data/processed/ID_to_city.csv')

city_lookup = pd.read_csv(LOOKUP_PATH)
city_lookup['City'] = city_lookup['City'].str.upper().str.strip()

city_aliases = {
    'BIDAR AIRPORT, KARN': 'BIDAR',
    'BIDAR AIRPORT, KARNATAKA': 'BIDAR',
    'COCHIN': 'KOCHI',
    'DEOGHAR AIRPORT': 'DEOGHAR',
    'GANGTOK': 'PAKYONG',
    'GONDIA AIRPORT': 'GONDIA',
    'HOLLONGI AIRPORT, ITANAGAR': 'ITANAGAR',
    'KALABURAGI, KARNAT': 'KALABURAGI',
    'KALABURAGI, KARNATAKA': 'KALABURAGI',
    'KUSHINAGAR': 'KUSHINAGAR INTERNATIONAL AIRPORT',
}

fallback_city_coordinates = {
    'BIDAR': (17.9018898, 77.4596205),
    'HISSAR': (29.2396596, 75.8174505),
    'JALGAON': (20.8428827, 75.5261246),
    'KALABURAGI': (17.2408957, 76.7697439),
    'MUNDRA': (22.8393153, 69.7249477),
    'NANDED': (19.0942729, 77.4832628),
    'PATHANKOT': (32.3021899, 75.6589406),
    'PITHORAGARH': (29.585871, 80.215167),
    'PORBANDAR': (21.6029751, 69.8538067),
    'PUDUCHERRY': (10.9156489, 79.8069488),
}

for year_data_group in list(yearly_data.values()) + list(yearly_data_graphi_format.values()):
    for month_data in year_data_group:
        month_data['CITY 1'] = month_data['CITY 1'].replace(city_aliases)
        month_data['CITY 2'] = month_data['CITY 2'].replace(city_aliases)

existing_lookup_cities = set(city_lookup['City'])
all_notebook_cities = sorted(
    {
        city
        for year_data_group in list(yearly_data.values()) + list(yearly_data_graphi_format.values())
        for month_data in year_data_group
        for city in pd.concat([month_data['CITY 1'], month_data['CITY 2']]).dropna().astype(str).str.upper().str.strip().unique()
    }
)
missing_lookup_cities = [city for city in all_notebook_cities if city not in existing_lookup_cities]

if missing_lookup_cities:
    next_id = int(city_lookup['ID'].max()) + 1
    missing_rows = []
    for city in missing_lookup_cities:
        if city not in fallback_city_coordinates:
            raise KeyError(f'Missing coordinates for city: {city}')
        latitude, longitude = fallback_city_coordinates[city]
        missing_rows.append({
            'ID': next_id,
            'City': city,
            'Latitude': latitude,
            'Longitude': longitude,
        })
        next_id += 1
    city_lookup = pd.concat([city_lookup, pd.DataFrame(missing_rows)], ignore_index=True)

place_to_ID_dict = dict(zip(city_lookup['City'], city_lookup['ID']))
ID_to_place_dict = dict(zip(city_lookup['ID'], city_lookup['City']))
places_coordinates = dict(zip(city_lookup['City'], zip(city_lookup['Latitude'], city_lookup['Longitude'])))

data_for_ID = pd.concat(
    [yearly_data_graphi_format[year][-1] for year in sorted(YEAR_MONTHS, reverse=True)],
    ignore_index=True,
)
data_for_ID = data_for_ID.groupby(['CITY 1', 'CITY 2'], as_index=False).sum()

for year_data_group in list(yearly_data.values()) + list(yearly_data_graphi_format.values()):
    for month_data in year_data_group:
        month_data['CITY 1 ID'] = month_data['CITY 1'].map(place_to_ID_dict)
        month_data['CITY 2 ID'] = month_data['CITY 2'].map(place_to_ID_dict)
        month_data['LATITUDE 1'] = month_data['CITY 1'].map(lambda city: places_coordinates[city][0])
        month_data['LONGITUDE 1'] = month_data['CITY 1'].map(lambda city: places_coordinates[city][1])
        month_data['LATITUDE 2'] = month_data['CITY 2'].map(lambda city: places_coordinates[city][0])
        month_data['LONGITUDE 2'] = month_data['CITY 2'].map(lambda city: places_coordinates[city][1])
        month_data['GEOMETRY CITY 1'] = [Point(xy) for xy in zip(month_data['LONGITUDE 1'], month_data['LATITUDE 1'])]
        month_data['GEOMETRY CITY 2'] = [Point(xy) for xy in zip(month_data['LONGITUDE 2'], month_data['LATITUDE 2'])]

len(places_coordinates)

### Making the Networks


In [ ]:
EDGE_COLUMNS = ['PASSENGERS TO CITY 2', 'GEOMETRY CITY 1', 'GEOMETRY CITY 2']


def build_network(dataframe, name):
    network = nx.from_pandas_edgelist(
        dataframe,
        'CITY 1',
        'CITY 2',
        EDGE_COLUMNS,
        create_using=nx.DiGraph(),
    )
    network.name = name
    return network


yearly_networks = {}
combined_network_lookup = {}
for year, months in YEAR_MONTHS.items():
    monthly_networks = []
    for idx, month in enumerate(months):
        network = build_network(yearly_data_graphi_format[year][idx], f"{MONTH_LABELS[month]} {year}")
        globals()[f'{month.lower()}{year}_network'] = network
        monthly_networks.append(network)

    combined_network = build_network(yearly_data_graphi_format[year][-1], f'Combined {year}')
    globals()[f'combined{year}_network'] = combined_network
    globals()[f'year{year}_networks'] = monthly_networks

    yearly_networks[year] = monthly_networks
    combined_network_lookup[year] = combined_network


## Extra Declarations


In [ ]:
def typical_passenger_flow(networks_list, city_name):
    for network in networks_list:
        if city_name in network.nodes:
            total_departed_passengers = sum(
                data['PASSENGERS TO CITY 2'] for _, _, data in network.out_edges(city_name, data=True)
            )
            total_arrived_passengers = sum(
                data['PASSENGERS TO CITY 2'] for _, _, data in network.in_edges(city_name, data=True)
            )
            print(f'{city_name} - {network.name} - {total_departed_passengers} - {total_arrived_passengers}')


month_networks_with_2015 = [
    network
    for year in YEAR_MONTHS
    for network in yearly_networks[year]
]
month_networks_with_2015_names = [
    f"{MONTH_LABELS[month]} {year}"
    for year, months in YEAR_MONTHS.items()
    for month in months
]

month_networks = [
    network
    for year in range(2016, 2024)
    for network in yearly_networks[year]
]
month_networks_names = [
    f"{MONTH_LABELS[month]} {year}"
    for year in range(2016, 2024)
    for month in YEAR_MONTHS[year]
]

combined_networks = [combined_network_lookup[year] for year in range(2016, 2024)]
combined_networks_names = [str(year) for year in range(2016, 2024)]
combined_all_data = [globals()[f'combined{year}_data'] for year in YEAR_MONTHS]

jan_networks = [globals()[f'jan{year}_network'] for year in range(2016, 2024)]
feb_networks = [globals()[f'feb{year}_network'] for year in range(2016, 2024)]
mar_networks = [globals()[f'mar{year}_network'] for year in range(2016, 2024)]
apr_networks = [globals()[f'apr{year}_network'] for year in range(2016, 2024)]
may_networks = [globals()[f'may{year}_network'] for year in range(2016, 2024)]
jun_networks = [globals()[f'jun{year}_network'] for year in range(2016, 2024)]
jul_networks = [globals()[f'jul{year}_network'] for year in range(2016, 2024)]
aug_networks = [globals()[f'aug{year}_network'] for year in range(2016, 2024)]
sep_networks = [globals()[f'sep{year}_network'] for year in range(2016, 2024)]
oct_networks = [globals()[f'oct{year}_network'] for year in range(2016, 2024)]
nov_networks = [globals()[f'nov{year}_network'] for year in range(2016, 2024)]
dec_networks = [globals()[f'dec{year}_network'] for year in range(2016, 2024)]

year_to_month_bounds = {}
cursor = 0
for year, months in YEAR_MONTHS.items():
    year_to_month_bounds[year] = (cursor, cursor + len(months))
    cursor += len(months)


def month_slice_for_years(start_year, end_year):
    start_index = year_to_month_bounds[start_year][0]
    end_index = year_to_month_bounds[end_year][1]
    return start_index, end_index


In [ ]:
all_cities = set(places_coordinates.keys())
tier1_cities = ['DELHI', 'MUMBAI', 'KOLKATA', 'CHENNAI', 'BENGALURU', 'HYDERABAD', 'PUNE', 'AHMEDABAD']
tier2_cities = ['JAIPUR', 'SURAT', 'LUCKNOW', 'KANPUR', 'NAGPUR', 'INDORE', 'COIMBATORE', 'KOCHI', 'VISAKHAPATNAM', 'BHOPAL', 'PATNA', 'AGRA', 'VADODARA', 'GHAZIABAD', 'VARANASI', 'SRINAGAR', 'AURANGABAD', 'AMRITSAR', 'ALLAHABAD', 'RANCHI', 'JABALPUR', 'GWALIOR', 'VIJAYAWADA', 'JODHPUR', 'RAIPUR', 'GUWAHATI', 'CHANDIGARH', 'HUBLI', 'BAREILLY', 'MYSORE', 'JAMSHEDPUR', 'BHAVNAGAR', 'DURGAPUR', 'ROURKELA', 'NANDED', 'KOLHAPUR', 'JALGAON', 'GOA', 'UJJAIN', 'SHIRDI', 'BHUBANESWAR', 'JAMNAGAR', 'JAMMU', 'TIRUCHIRAPALLY', 'DEHRADUN', 'BILASPUR', 'BIKANER', 'COCHI', 'UDAIPUR']
tier3_cities = ['DEOGHAR', 'SHILLONG', 'IMPHAL', 'DARBHANGA', 'GORAKHPUR', 'JALPAIGURI', 'NIZAMABAD', 'JUNAGADH', 'GANGTOK', 'JORHAT', 'BIJAPUR', 'AIZAWL', 'DIBRUGARH', 'SAGAR', 'SATNA', 'MUNGER', 'KAKCHING', 'MANGALORE', 'RAJKOT']
tier4_cities = sorted(all_cities - set(tier1_cities + tier2_cities + tier3_cities))

tier_cities = {
    'Tier1': tier1_cities,
    'Tier2': tier2_cities,
    'Tier3': tier3_cities,
    'Tier4': tier4_cities,
}

underserved_airports = [
    'AGARTALA', 'AIZAWL', 'BELGAUM', 'BHAVNAGAR', 'BHUJ', 'BIKANER', 'DIBRUGARH', 'DIMAPUR',
    'GORAKHPUR', 'HUBLI', 'IMPHAL', 'JODHPUR', 'JORHAT', 'KOLHAPUR', 'LILABARI', 'PASIGHAT',
    'SHILLONG', 'SILCHAR', 'TEZPUR'
]

unserved_airports = [
    'AGATTI ISLAND', 'BAREILLY', 'DARBHANGA', 'DEOGHAR', 'DURGAPUR', 'GAYA', 'JEYPORE', 'JHARSUGUDA',
    'KULLU', 'LEH', 'MOPA, GOA', 'RAJAHMUNDRY', 'SHIRDI', 'UTKELA', 'JAISALMER', 'MALVAN', 'ADAMPUR',
    'KESHOD', 'PITHORAGARH', 'GANGTOK', 'COCHIN', 'HISSAR', 'MUNDRA', 'JALGAON', 'PORBANDAR', 'NANDED',
    'BIDAR', 'SIMLA', 'RUPSI', 'ZERO AIRPORT', 'LUDHIANA', 'SALEM', 'AYODHYA INTERNATIONAL AIRPORT',
    'UTTARLAI', 'BIDAR AIRPORT, KARNATAKA', 'GONDIA AIRPORT', 'PAKYONG', 'KUSHINAGAR INTERNATIONAL AIRPORT'
]


## Extra Functions


In [ ]:
# region

# Find maximum number of passengers in any given month and year separately.
max_passengers_monthwise_by_year = {
    year: [max(dataframe['PASSENGERS TO CITY 2']) for dataframe in yearly_data_graphi_format[year][:-1]]
    for year in YEAR_MONTHS
}
max_passengers_by_year = {
    year: max(yearly_data_graphi_format[year][-1]['PASSENGERS TO CITY 2'])
    for year in YEAR_MONTHS
}

for year, values in max_passengers_monthwise_by_year.items():
    globals()[f'max_passengers_{year}_monthwise'] = values
for year, value in max_passengers_by_year.items():
    globals()[f'max_passengers_{year}'] = value

max_passengers_yearwise = max(max_passengers_by_year.values())
max_passengers_monthwise = max(
    value
    for values in max_passengers_monthwise_by_year.values()
    for value in values
)

# print('MAXIMUM NUMBER OF PASSENGERS IN ANY GIVEN MONTH AND YEAR IS: ', max_passengers_monthwise, max_passengers_yearwise)

# endregion

india = gpd.read_file(r'../references/maps_with_python/india-polygon.shp')


In [ ]:
def plot_network_categorically(network_year):
    startid, endid = year_to_month_bounds[network_year]

    min_thickness, max_thickness = 0.2, 100
    min_weight = 0
    max_weight = max_passengers_monthwise

    weight_bins = [0, 500, 1000, 2500, 5000, 10000, max_weight]
    cmap_colors = ['tab:gray', 'tab:green', 'tab:olive', 'tab:orange', 'tab:red', 'tab:purple']

    cmap = colors.ListedColormap(cmap_colors)
    norm = colors.BoundaryNorm(weight_bins, cmap.N)

    fig, ax = plt.subplots(4, 3, figsize=(30, 45))
    for i, (graph, name) in tqdm(
        enumerate(zip(month_networks_with_2015[startid:endid], month_networks_with_2015_names[startid:endid]))
    ):
        india.plot(ax=ax[i // 3][i % 3], color='gray', edgecolor='black', alpha=0.5)
        for u, v, passengerstocity2 in graph.edges(data=True):
            ax[i // 3][i % 3].plot(places_coordinates[u][1], places_coordinates[u][0], 'ro', markersize=5)
            ax[i // 3][i % 3].plot(places_coordinates[v][1], places_coordinates[v][0], 'ro', markersize=5)
            start = places_coordinates[u]
            end = places_coordinates[v]
            weight = passengerstocity2['PASSENGERS TO CITY 2']
            thickness = min_thickness + ((weight - min_weight) / (max_weight - min_weight)) * max_thickness
            color = cmap(norm(weight))
            ax[i // 3][i % 3].arrow(
                start[1],
                start[0],
                end[1] - start[1],
                end[0] - start[0],
                alpha=0.8,
                length_includes_head=True,
                fc=color,
                ec=color,
            )
            ax[i // 3][i % 3].set_title(name)
            legend_elements = [
                mpl.patches.Rectangle((0, 0), 1, 1, facecolor=cmap_colors[idx], label=f'{weight_bins[idx]} - {weight_bins[idx + 1]}')
                for idx in range(len(weight_bins) - 1)
            ]
            ax[i // 3][i % 3].legend(handles=legend_elements, title='Edge Weight Categories')

    plt.show()


In [ ]:
def plot_network(network_year):
    startid, endid = year_to_month_bounds[network_year]

    min_thickness, max_thickness = 0.2, 100
    min_weight = 0
    max_weight = max_passengers_monthwise

    cmap = plt.cm.viridis
    norm = plt.Normalize(vmin=min_weight, vmax=max_weight * 1.3)

    fig, ax = plt.subplots(4, 3, figsize=(30, 45))
    for i, (graph, name) in tqdm(
        enumerate(zip(month_networks_with_2015[startid:endid], month_networks_with_2015_names[startid:endid]))
    ):
        india.plot(ax=ax[i // 3][i % 3], color='gray', edgecolor='black', alpha=0.5)
        for u, v, passengerstocity2 in graph.edges(data=True):
            ax[i // 3][i % 3].plot(places_coordinates[u][1], places_coordinates[u][0], 'ro', markersize=5)
            ax[i // 3][i % 3].plot(places_coordinates[v][1], places_coordinates[v][0], 'ro', markersize=5)
            start = places_coordinates[u]
            end = places_coordinates[v]
            weight = passengerstocity2['PASSENGERS TO CITY 2']
            thickness = min_thickness + ((weight - min_weight) / (max_weight - min_weight)) * max_thickness
            color = cmap(norm(weight * 100))
            ax[i // 3][i % 3].arrow(
                start[1],
                start[0],
                end[1] - start[1],
                end[0] - start[0],
                alpha=0.8,
                length_includes_head=True,
                fc=color,
                ec=color,
            )
            ax[i // 3][i % 3].set_title(name)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm._A = []
    plt.colorbar(sm, ax=ax.ravel().tolist())
    plt.show()


In [ ]:
def nodes_by_degree(G):
    # This function will return nodes sorted by their degree (sum of in-degree and out-degree for directed graphs)
    return sorted(G.nodes(), key=lambda x: G.degree(x), reverse=True)

def plot_cities_passengers_in_time_via_rank_helper(networks_list, model_network, startId, endId):
    departures, arrivals = [], []
    list_names = nodes_by_degree(model_network)[startId:endId]

    for G in networks_list:
        dict = {city: [0, 0] for city in list_names} # Total departed and arrived passengers
        for city1, city2, data in G.edges(data=True):
            if city1 in dict:
                dict[city1][0] += data['PASSENGERS TO CITY 2']
            if city2 in dict:
                dict[city2][1] += data['PASSENGERS TO CITY 2']
    
        for city in dict:
            departures.append(dict[city][0])
            arrivals.append(dict[city][1])

    return departures, arrivals, list_names

def plot_cities_passengers_in_time_via_rank(networks_list, network_names, model_network, startId, endId, figsize=(30, 10), fontsize=7):
    departures, arrivals, list_names = plot_cities_passengers_in_time_via_rank_helper(networks_list, model_network, startId, endId)
    numberOfCities = endId - startId 
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    length = len(departures)//(numberOfCities)
    
    print(f"Plotted {numberOfCities} cities!")
    for i, name in enumerate(list_names):
        ax[0].scatter(np.arange(length), departures[i::numberOfCities]); ax[1].scatter(np.arange(length), arrivals[i::numberOfCities])
        ax[0].plot(np.arange(length), departures[i::numberOfCities], label=f'{name} Departures'); ax[1].plot(np.arange(length), arrivals[i::numberOfCities], label=f'{name} Arrivals')
    
    ax[0].set_xticks(np.arange(length)); ax[1].set_xticks(np.arange(length))
    ax[0].set_xticklabels(network_names); ax[1].set_xticklabels(network_names)
    ax[0].tick_params(axis='x', rotation=90); ax[1].tick_params(axis='x', rotation=90)
    ax[0].set_xlabel('Time'); ax[1].set_xlabel('Time')
    ax[0].set_ylabel('Number of Passengers'); ax[1].set_ylabel('Number of Passengers')
    ax[0].set_title('Number of Passengers Departed with Time'); ax[1].set_title('Number of Passengers Arrived with Time')
    ax[0].legend(fontsize=fontsize); ax[1].legend(fontsize=fontsize)
    ylim1 = ax[0].get_ylim()
    ylim2 = ax[1].get_ylim()
    ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)
    
    plt.show()

In [ ]:
def plot_cities_passengers_in_time_via_tier_helper(networks_list, startId, endId, tier):
    departures, arrivals = [], []
    list_names = tier_cities[tier][startId:endId]

    for G in networks_list:
        dict = {city: [0, 0] for city in list_names} # Total departed and arrived passengers
        for city1, city2, data in G.edges(data=True):
            if city1 in dict:
                dict[city1][0] += data['PASSENGERS TO CITY 2']
            if city2 in dict:
                dict[city2][1] += data['PASSENGERS TO CITY 2']
    
        for city in dict:
            departures.append(dict[city][0])
            arrivals.append(dict[city][1])

    return departures, arrivals, list_names

def plot_cities_passengers_in_time_via_tier(networks_list, network_names, startId=0, endId=-1, tier='Tier3', figsize=(30, 10), fontsize=7):
    departures, arrivals, list_names = plot_cities_passengers_in_time_via_tier_helper(networks_list, startId, endId, tier)
    numberOfCities = endId - startId 
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    length = len(departures)//(numberOfCities)
    print(f"Plotted {numberOfCities} cities!")
    for i, name in enumerate(list_names):
        ax[0].scatter(np.arange(length), departures[i::numberOfCities]); ax[1].scatter(np.arange(length), arrivals[i::numberOfCities])
        ax[0].plot(np.arange(length), departures[i::numberOfCities], label=f'{name} Departures'); ax[1].plot(np.arange(length), arrivals[i::numberOfCities], label=f'{name} Arrivals')
    
    ax[0].set_xticks(np.arange(length)); ax[1].set_xticks(np.arange(length))
    ax[0].set_xticklabels(network_names); ax[1].set_xticklabels(network_names)
    ax[0].tick_params(axis='x', rotation=90); ax[1].tick_params(axis='x', rotation=90)
    ax[0].set_xlabel('Time'); ax[1].set_xlabel('Time')
    ax[0].set_ylabel('Number of Passengers'); ax[1].set_ylabel('Number of Passengers')
    ax[0].set_title('Number of Passengers Departed with Time'); ax[1].set_title('Number of Passengers Arrived with Time')
    ax[0].legend(fontsize=fontsize); ax[1].legend(fontsize=fontsize)
    ylim1 = ax[0].get_ylim()
    ylim2 = ax[1].get_ylim()
    ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)

    plt.show()

In [ ]:
def find_critical_nodes_directed(G):
    G_temp = deepcopy(G)  # Create a copy of the network to manipulate without altering the original network
    initial_size = len(max(nx.strongly_connected_components(G_temp), key=len))  # Initial size of the largest strongly connected component

    loss_dict = {}  # Dictionary to store loss for each node

    for node in G_temp.nodes():
        # Temporarily remove the node
        G_removed = deepcopy(G_temp)
        G_removed.remove_node(node)

        if nx.number_strongly_connected_components(G_removed) > 0:
            # Calculate the new size of the largest strongly connected component
            largest_scc_size = len(max(nx.strongly_connected_components(G_removed), key=len))
            loss = (initial_size - largest_scc_size) / initial_size
        else:
            # If removing the node disconnects the network completely
            loss = 1

        # Store the loss for the current node
        loss_dict[node] = loss

    # Sort nodes based on loss from maximum to minimum
    sorted_nodes = sorted(loss_dict.keys(), key=lambda x: loss_dict[x], reverse=True)

    return sorted_nodes

def plot_cities_passengers_in_time_via_connectivity_helper(networks_list, startId, endId):
    departures, arrivals = [], []
    list_names = find_critical_nodes_directed(networks_list[0])[startId:endId]

    for G in networks_list:
        dict = {city: [0, 0] for city in list_names} # Total departed and arrived passengers
        for city1, city2, data in G.edges(data=True):
            if city1 in dict:
                dict[city1][0] += data['PASSENGERS TO CITY 2']
            if city2 in dict:
                dict[city2][1] += data['PASSENGERS TO CITY 2']
    
        for city in dict:
            departures.append(dict[city][0])
            arrivals.append(dict[city][1])

    return departures, arrivals, list_names

def plot_cities_passengers_in_time_via_connectivity(networks_list, network_names, startId, endId, figsize=(30, 10), fontsize=7):
    departures, arrivals, list_names = plot_cities_passengers_in_time_via_connectivity_helper(networks_list, startId, endId)
    numberOfCities = endId - startId 
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    length = len(departures)//(numberOfCities)
    print(f"Plotted {numberOfCities} cities!")
    for i, name in enumerate(list_names):
        ax[0].scatter(np.arange(length), departures[i::numberOfCities]); ax[1].scatter(np.arange(length), arrivals[i::numberOfCities])
        ax[0].plot(np.arange(length), departures[i::numberOfCities], label=f'{name} Departures'); ax[1].plot(np.arange(length), arrivals[i::numberOfCities], label=f'{name} Arrivals')
    
    ax[0].set_xticks(np.arange(length)); ax[1].set_xticks(np.arange(length))
    ax[0].set_xticklabels(network_names); ax[1].set_xticklabels(network_names)
    ax[0].tick_params(axis='x', rotation=90); ax[1].tick_params(axis='x', rotation=90)
    ax[0].set_xlabel('Time'); ax[1].set_xlabel('Time')
    ax[0].set_ylabel('Number of Passengers'); ax[1].set_ylabel('Number of Passengers')
    ax[0].set_title('Number of Passengers Departed with Time'); ax[1].set_title('Number of Passengers Arrived with Time')
    ax[0].legend(fontsize=fontsize); ax[1].legend(fontsize=fontsize)
    ylim1 = ax[0].get_ylim()
    ylim2 = ax[1].get_ylim()
    ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)

    plt.show()

In [ ]:
def plot_underserved_cities_passengers_in_time_helper(networks_list, startId, endId):
    departures, arrivals = [], []
    list_names = underserved_airports[startId:endId]

    for G in networks_list:
        dict = {city: [0, 0] for city in list_names} # Total departed and arrived passengers
        for city1, city2, data in G.edges(data=True):
            if city1 in dict:
                dict[city1][0] += data['PASSENGERS TO CITY 2']
            if city2 in dict:
                dict[city2][1] += data['PASSENGERS TO CITY 2']
    
        for city in dict:
            departures.append(dict[city][0])
            arrivals.append(dict[city][1])

    return departures, arrivals, list_names

def plot_underserved_cities_passengers_in_time(networks_list, network_names, startId, endId, figsize=(30, 10), fontsize=7):
    departures, arrivals, list_names = plot_underserved_cities_passengers_in_time_helper(networks_list, startId, endId)
    numberOfCities = endId - startId 
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    length = len(departures)//(numberOfCities)
    print(f"Plotted {numberOfCities} cities!")
    for i, name in enumerate(list_names):
        ax[0].scatter(np.arange(length), departures[i::numberOfCities]); ax[1].scatter(np.arange(length), arrivals[i::numberOfCities])
        ax[0].plot(np.arange(length), departures[i::numberOfCities], label=f'{name} Departures'); ax[1].plot(np.arange(length), arrivals[i::numberOfCities], label=f'{name} Arrivals')
    
    ax[0].set_xticks(np.arange(length)); ax[1].set_xticks(np.arange(length))
    ax[0].set_xticklabels(network_names); ax[1].set_xticklabels(network_names)
    ax[0].tick_params(axis='x', rotation=90); ax[1].tick_params(axis='x', rotation=90)
    ax[0].set_xlabel('Time'); ax[1].set_xlabel('Time')
    ax[0].set_ylabel('Number of Passengers'); ax[1].set_ylabel('Number of Passengers')
    ax[0].set_title('Number of Passengers Departed with Time'); ax[1].set_title('Number of Passengers Arrived with Time')
    ax[0].legend(fontsize=fontsize); ax[1].legend(fontsize=fontsize)
    ylim1 = ax[0].get_ylim()
    ylim2 = ax[1].get_ylim()
    ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)

    plt.show()

In [ ]:
def plot_unserved_cities_passengers_in_time_helper(networks_list, startId, endId):
    departures, arrivals = [], []
    list_names = unserved_airports[startId:endId]

    for G in networks_list:
        dict = {city: [0, 0] for city in list_names} # Total departed and arrived passengers
        for city1, city2, data in G.edges(data=True):
            if city1 in dict:
                dict[city1][0] += data['PASSENGERS TO CITY 2']
            if city2 in dict:
                dict[city2][1] += data['PASSENGERS TO CITY 2']
    
        for city in dict:
            departures.append(dict[city][0])
            arrivals.append(dict[city][1])

    return departures, arrivals, list_names

def plot_unserved_cities_passengers_in_time(networks_list, network_names, startId, endId, figsize=(30, 10), fontsize=7):
    departures, arrivals, list_names = plot_unserved_cities_passengers_in_time_helper(networks_list, startId, endId)
    numberOfCities = endId - startId 
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    length = len(departures)//(numberOfCities)
    print(f"Plotted {numberOfCities} cities!")
    for i, name in enumerate(list_names):
        ax[0].scatter(np.arange(length), departures[i::numberOfCities]); ax[1].scatter(np.arange(length), arrivals[i::numberOfCities])
        ax[0].plot(np.arange(length), departures[i::numberOfCities], label=f'{name} Departures'); ax[1].plot(np.arange(length), arrivals[i::numberOfCities], label=f'{name} Arrivals')
    
    ax[0].set_xticks(np.arange(length)); ax[1].set_xticks(np.arange(length))
    ax[0].set_xticklabels(network_names); ax[1].set_xticklabels(network_names)
    ax[0].tick_params(axis='x', rotation=90); ax[1].tick_params(axis='x', rotation=90)
    ax[0].set_xlabel('Time'); ax[1].set_xlabel('Time')
    ax[0].set_ylabel('Number of Passengers'); ax[1].set_ylabel('Number of Passengers')
    ax[0].set_title('Number of Passengers Departed with Time'); ax[1].set_title('Number of Passengers Arrived with Time')
    ax[0].legend(fontsize=fontsize); ax[1].legend(fontsize=fontsize)
    ylim1 = ax[0].get_ylim()
    ylim2 = ax[1].get_ylim()
    ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)

    plt.show()

In [ ]:
def plot_selected_cities_passengers_in_time_helper(networks_list, startId, endId, selected_cities):
    departures, arrivals = [], []
    list_names = selected_cities[startId:endId]

    for G in networks_list:
        dict = {city: [0, 0] for city in list_names} # Total departed and arrived passengers
        for city1, city2, data in G.edges(data=True):
            if city1 in dict:
                dict[city1][0] += data['PASSENGERS TO CITY 2']
            if city2 in dict:
                dict[city2][1] += data['PASSENGERS TO CITY 2']
    
        for city in dict:
            departures.append(dict[city][0])
            arrivals.append(dict[city][1])

    return departures, arrivals, list_names

def plot_selected_cities_passengers_in_time(networks_list, network_names, startId, endId, selected_cities, figsize=(30, 10), fontsize=7):
    departures, arrivals, list_names = plot_selected_cities_passengers_in_time_helper(networks_list, startId, endId, selected_cities)
    numberOfCities = endId - startId 
    
    fig, ax = plt.subplots(1, 2, figsize=figsize)
    length = len(departures)//(numberOfCities)
    print(f"Plotted {numberOfCities} cities!")
    for i, name in enumerate(list_names):
        ax[0].scatter(np.arange(length), departures[i::numberOfCities]); ax[1].scatter(np.arange(length), arrivals[i::numberOfCities])
        ax[0].plot(np.arange(length), departures[i::numberOfCities], label=f'{name} Departures'); ax[1].plot(np.arange(length), arrivals[i::numberOfCities], label=f'{name} Arrivals')
    
    ax[0].set_xticks(np.arange(length)); ax[1].set_xticks(np.arange(length))
    ax[0].set_xticklabels(network_names); ax[1].set_xticklabels(network_names)
    ax[0].tick_params(axis='x', rotation=90); ax[1].tick_params(axis='x', rotation=90)
    ax[0].set_xlabel('Time'); ax[1].set_xlabel('Time')
    ax[0].set_ylabel('Number of Passengers'); ax[1].set_ylabel('Number of Passengers')
    ax[0].set_title('Number of Passengers Departed with Time'); ax[1].set_title('Number of Passengers Arrived with Time')
    ax[0].legend(fontsize=fontsize); ax[1].legend(fontsize=fontsize)
    ylim1 = ax[0].get_ylim()
    ylim2 = ax[1].get_ylim()
    ylim = (min(ylim1[0], ylim2[0]), max(ylim1[1], ylim2[1]))
    ax[0].set_ylim(ylim); ax[1].set_ylim(ylim)

    plt.show()

## Documentation


* Individual month data can be accessed like `{month}{year}_data`   
  
* `combined{year}_data`  is all months of a year data can be accessed to  

* `year{year}_data` = all month data + `combined{year}_data`

* `year{year}_data_graphi_format` = graphi format of `year{year}_data`

* `{month}{year}_network` = network of individual month data

* `combined{year}_network` = network of all months of a year

* `month_networks` = list of all month networks in order

* `month_network_names` = list of all month network names in order

* `year{year}_networks` = network of all months of a year + `combined{year}_network`

* `{month}_networks` = list of all month networks in different years

* `combined_networks` = list of all combined networks in different years

* `place_to_ID_dict` = dictionary of place string to ID

* `ID_to_place_dict` = dictionary of ID to place string

* `places_coordinates` = dictionary of place string to coordinates

## Actual Work


In [ ]:
# Find number of unique airports in the dataset.
unique_airports = set()
for dataframe in combined_all_data:
    unique_airports.update(set(dataframe['CITY 1']))
    unique_airports.update(set(dataframe['CITY 2']))

len(unique_airports)


### Plotting networks category wise

In [ ]:
# plot_network_categorically(2015)

In [ ]:
# plot_network_categorically(2016)

In [ ]:
# plot_network_categorically(2017)

In [ ]:
# plot_network_categorically(2018)

In [ ]:
# plot_network_categorically(2019)

In [ ]:
# plot_network_categorically(2020)

In [ ]:
# plot_network_categorically(2021)

In [ ]:
# plot_network_categorically(2022)

In [ ]:
# plot_network_categorically(2023)

### Plotting networks via continuous color

In [ ]:
# plot_network(2015)

In [ ]:
# plot_network(2016)

In [ ]:
# plot_network(2017)

In [ ]:
# plot_network(2018)

In [ ]:
# plot_network(2019)

In [ ]:
# plot_network(2020)

In [ ]:
# plot_network(2021)

In [ ]:
# plot_network(2022)

In [ ]:
# plot_network(2023)

### Plotting Passengers Flow through High Degree Cities

In [ ]:
startYear = 2015
endYear = 2018
startIndex, endIndex = month_slice_for_years(startYear, endYear)
plot_cities_passengers_in_time_via_rank(
    month_networks_with_2015[startIndex:endIndex],
    month_networks_with_2015_names[startIndex:endIndex],
    combined2016_network,
    0,
    10,
)

num = 0
if num != 0:
    num = num - 5
plot_cities_passengers_in_time_via_rank(
    combined_networks,
    combined_networks_names,
    combined2016_network,
    num,
    num + 5,
)
num = num + 5


### Plotting Passengers Flow on the basis of Tier

In [ ]:
startYear = 2015
endYear = 2023
tier1length = len(tier_cities['Tier1'])
tier2length = len(tier_cities['Tier2'])
tier3length = len(tier_cities['Tier3'])
tier4length = len(tier_cities['Tier4'])
print('Tier1 Length: ', tier1length, 'Tier2 Length: ', tier2length, 'Tier3 Length: ', tier3length, 'Tier4 Length: ', tier4length)

startIndex, endIndex = month_slice_for_years(startYear, endYear)
plot_cities_passengers_in_time_via_tier(
    month_networks_with_2015[startIndex:endIndex],
    month_networks_with_2015_names[startIndex:endIndex],
    5,
    10,
    'Tier4',
)

num = 0
if num != 0:
    num = num - 5
plot_cities_passengers_in_time_via_tier(
    combined_networks,
    combined_networks_names,
    num,
    num + 5,
    tier='Tier4',
    fontsize=10,
)
num = num + 5


### Plotting Passenger Flow on the basis of Connectivity

In [ ]:
startYear = 2016
endYear = 2018
startIndex, endIndex = month_slice_for_years(startYear, endYear)
plot_cities_passengers_in_time_via_connectivity(
    month_networks_with_2015[startIndex:endIndex],
    month_networks_with_2015_names[startIndex:endIndex],
    0,
    15,
)

num = 0
if num != 0:
    num = num - 5
plot_cities_passengers_in_time_via_connectivity(
    combined_networks,
    combined_networks_names,
    0,
    15,
    fontsize=10,
)
num = num + 5


### Plotting Passenger Flow in Underserved Cities

In [ ]:
startYear = 2016
endYear = 2018
maxLen = len(underserved_airports)
startIndex, endIndex = month_slice_for_years(startYear, endYear)
plot_underserved_cities_passengers_in_time(
    month_networks_with_2015[startIndex:endIndex],
    month_networks_with_2015_names[startIndex:endIndex],
    0,
    maxLen,
)

num = 0
if num != 0:
    num = num - 5
plot_underserved_cities_passengers_in_time(
    combined_networks,
    combined_networks_names,
    0,
    maxLen,
    fontsize=10,
)
num = num + 5


### Plotting Passenger Flow in Unserved Cities

In [ ]:
startYear = 2016
endYear = 2018
maxLen = len(unserved_airports)
startIndex, endIndex = month_slice_for_years(startYear, endYear)
plot_unserved_cities_passengers_in_time(
    month_networks_with_2015[startIndex:endIndex],
    month_networks_with_2015_names[startIndex:endIndex],
    0,
    maxLen,
)

num = 0
if num != 0:
    num = num - 5
plot_unserved_cities_passengers_in_time(
    combined_networks,
    combined_networks_names,
    0,
    maxLen,
    fontsize=8,
)
num = num + 5


### Next

In [ ]:
selected_cities = ['TRIVANDRUM', 'GORAKHPUR', 'IMPHAL', 'JORHAT', 'MANGALORE', 'SILCHAR', 'RAJAHMUNDRY', 'LEH', 'KADAPA', 'BAGDOGRA', 'KOZHIKODE', 'JAISALMER', 'TUTICORIN', 'PORBANDAR', 'AJMER', 'DEHRADUN', 'TIRUPATI', 'VIDYANAGAR', 'KANDLA', 'AGARTALA', 'CUDDAPAH', 'NASIK', 'SALEM', 'PORT BLAIR', 'KANNUR', 'PATHANKOT']

In [ ]:
plot_selected_cities_passengers_in_time(combined_networks[:4], combined_networks_names[:4], 0, 5, selected_cities, fontsize=10)
plot_selected_cities_passengers_in_time(combined_networks[:4], combined_networks_names[:4], 5, 10, selected_cities, fontsize=10)
plot_selected_cities_passengers_in_time(combined_networks[:4], combined_networks_names[:4], 10, 15, selected_cities, fontsize=10)
plot_selected_cities_passengers_in_time(combined_networks[:4], combined_networks_names[:4], 15, 20, selected_cities, fontsize=10)

In [ ]:
# Find increasing ratio of passengers from the selected_cities list
def find_arrived_passenger_dict(networks_list, selected_cities):
    dict = {city: [0 for i in range(len(networks_list))] for city in selected_cities} # Total arrived passengers
    for i, G in enumerate(networks_list):
        for city1, city2, data in G.edges(data=True):
            if city2 in dict:
                dict[city2][i] += data['PASSENGERS TO CITY 2']
            
    return dict

In [ ]:
arrived_passenger_dict = find_arrived_passenger_dict(combined_networks, selected_cities)

In [ ]:
for city in selected_cities:
    print(f"City: {arrived_passenger_dict[city][3]/arrived_passenger_dict[city][1] if arrived_passenger_dict[city][1] != 0 else 0} {city}")

### Network Diagnostics and Community Detection


In [ ]:
analysis_networks = {year: combined_network_lookup[year] for year in YEAR_MONTHS}


def plot_degree_distributions(network_map):
    fig, axes = plt.subplots(3, 3, figsize=(18, 12))
    for ax, (year, graph) in zip(axes.flatten(), network_map.items()):
        degrees = [degree for _, degree in graph.degree()]
        bin_count = min(20, max(5, len(set(degrees))))
        ax.hist(degrees, bins=bin_count)
        ax.set_title(str(year))
        ax.set_xlabel('Degree')
        ax.set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()


def fit_power_law_summary(network_map):
    rows = []
    for year, graph in network_map.items():
        degree_sequence = sorted([degree for _, degree in graph.degree() if degree > 0], reverse=True)
        if len(degree_sequence) < 2:
            continue
        fit = powerlaw.Fit(degree_sequence, verbose=False)
        rows.append({
            'Year': year,
            'Alpha': fit.power_law.alpha,
            'KS distance': fit.power_law.KS,
            'Min degree used': fit.power_law.xmin,
        })
    return pd.DataFrame(rows).sort_values('Year').reset_index(drop=True)


def simulate_network_disintegration_directed(graph, mode='random'):
    # This function simulates network disintegration under targeted and random removal.
    # We see which fraction of the largest connected component remains after removing nodes.
    graph_copy = graph.copy()
    initial_size = len(max(nx.weakly_connected_components(graph_copy), key=len))
    fractions = [1.0]

    nodes = list(graph_copy.nodes())
    if mode == 'targeted':
        nodes = sorted(nodes, key=lambda node: graph_copy.in_degree(node) + graph_copy.out_degree(node), reverse=True)
    else:
        np.random.shuffle(nodes)

    for node in nodes:
        graph_copy.remove_node(node)
        if nx.number_weakly_connected_components(graph_copy) > 0:
            largest_component = len(max(nx.weakly_connected_components(graph_copy), key=len))
            fractions.append(largest_component / initial_size)
        else:
            fractions.append(0.0)
            break

    return fractions


def summarize_centrality(network_map):
    rows = []
    for year, graph in network_map.items():
        degree_centrality = nx.degree_centrality(graph)
        betweenness_centrality = nx.betweenness_centrality(graph)
        closeness_centrality = nx.closeness_centrality(graph)
        rows.append({
            'Year': year,
            'Top degree city': max(degree_centrality, key=degree_centrality.get),
            'Top degree centrality': max(degree_centrality.values()),
            'Top betweenness city': max(betweenness_centrality, key=betweenness_centrality.get),
            'Top betweenness centrality': max(betweenness_centrality.values()),
            'Top closeness city': max(closeness_centrality, key=closeness_centrality.get),
            'Top closeness centrality': max(closeness_centrality.values()),
        })
    return pd.DataFrame(rows).sort_values('Year').reset_index(drop=True)


def merge_communities(partition, target_number):
    # This function assumes a simple heuristic to merge communities until the target number is reached.
    community_to_nodes = collections.defaultdict(list)
    for node, community in partition.items():
        community_to_nodes[community].append(node)

    communities = list(community_to_nodes.keys())
    while len(communities) > target_number:
        communities.sort(key=lambda community: len(community_to_nodes[community]))
        smaller = communities.pop(0)
        next_smaller = communities.pop(0)
        for node in community_to_nodes[smaller]:
            partition[node] = next_smaller
        community_to_nodes[next_smaller].extend(community_to_nodes[smaller])
        del community_to_nodes[smaller]

    return partition


def summarize_communities(network_map, target_number=5):
    partitions = {}
    merged_partitions = {}
    rows = []
    for year, graph in network_map.items():
        partition = community_louvain.best_partition(graph.to_undirected())
        merged_partition = merge_communities(partition.copy(), target_number)
        partitions[year] = partition
        merged_partitions[year] = merged_partition
        counts = collections.Counter(merged_partition.values())
        rows.append({
            'Year': year,
            'Communities after merge': len(counts),
            'Largest merged community size': max(counts.values()),
        })
    summary = pd.DataFrame(rows).sort_values('Year').reset_index(drop=True)
    return partitions, merged_partitions, summary


def community_strengths(graph, partition):
    intra_community_strength = {}
    inter_community_strength = {}

    for node in graph.nodes():
        intra_community_strength[node] = 0
        inter_community_strength[node] = 0
        node_community = partition[node]

        for neighbor, data in graph[node].items():
            edge_weight = data.get('PASSENGERS TO CITY 2', 1)
            if partition[neighbor] == node_community:
                intra_community_strength[node] += edge_weight
            else:
                inter_community_strength[node] += edge_weight

    community_strength_df = pd.DataFrame({
        'City': list(graph.nodes()),
        'Intra-community strength': [intra_community_strength[node] for node in graph.nodes()],
        'Inter-community strength': [inter_community_strength[node] for node in graph.nodes()],
        'Community': [partition[node] for node in graph.nodes()],
    })
    return community_strength_df.sort_values('Inter-community strength', ascending=False).reset_index(drop=True)


def plot_community_map(graph, partition, title):
    community_ids = sorted(set(partition.values()))
    cmap = plt.cm.get_cmap('tab20', len(community_ids))
    community_to_color = {
        community_id: cmap(idx)
        for idx, community_id in enumerate(community_ids)
    }

    fig, ax = plt.subplots(figsize=(10, 15))
    india.plot(ax=ax, color='gray', edgecolor='black', alpha=0.5)
    for node in graph.nodes():
        latitude, longitude = places_coordinates[node]
        ax.plot(longitude, latitude, 'o', markersize=8, color=community_to_color[partition[node]])

    for u, v, passengerstocity2 in graph.edges(data=True):
        start = places_coordinates[u]
        end = places_coordinates[v]
        weight = passengerstocity2['PASSENGERS TO CITY 2']
        alpha = min(0.8, 0.05 + (weight / max_passengers_yearwise))
        ax.arrow(
            start[1],
            start[0],
            end[1] - start[1],
            end[0] - start[0],
            alpha=alpha,
            length_includes_head=True,
            fc=community_to_color[partition[u]],
            ec=community_to_color[partition[u]],
        )

    ax.set_title(title)
    plt.show()


In [ ]:
plot_degree_distributions(analysis_networks)


In [ ]:
power_law_summary = fit_power_law_summary(analysis_networks)
power_law_summary


In [ ]:
robustness_years = [2015, 2020, 2023]

fig, axes = plt.subplots(1, len(robustness_years), figsize=(18, 5))
for ax, year in zip(axes, robustness_years):
    random_curve = simulate_network_disintegration_directed(analysis_networks[year], mode='random')
    targeted_curve = simulate_network_disintegration_directed(analysis_networks[year], mode='targeted')

    ax.plot(np.linspace(0, 1, len(random_curve)), random_curve, label='Random Removal', marker='o')
    ax.plot(np.linspace(0, 1, len(targeted_curve)), targeted_curve, label='Targeted Removal', marker='x')
    ax.set_title(f'Network Robustness: {year}')
    ax.set_xlabel('Fraction of Nodes Removed')
    ax.set_ylabel('Fraction of Largest Connected Component')
    ax.grid(True)
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
centrality_summary = summarize_centrality(analysis_networks)
centrality_summary


In [ ]:
partitions_by_year, merged_partitions_by_year, community_summary = summarize_communities(analysis_networks, target_number=5)
community_summary


In [ ]:
example_year = 2023
plot_community_map(
    combined_network_lookup[example_year],
    merged_partitions_by_year[example_year],
    f'{example_year} Combined Network Communities',
)

community_strength_summary = community_strengths(
    combined_network_lookup[example_year],
    merged_partitions_by_year[example_year],
)
community_strength_summary.head(10)
